In [2]:
import os
import re
import psycopg2
from datetime import datetime
from google.api_core.client_options import ClientOptions
from google.cloud import documentai_v1 as documentai
from google.cloud import aiplatform
import vertexai
from vertexai.language_models import TextEmbeddingModel
from process_document_sample import process_document_sample

c:\Users\sakst\.conda\envs\vestas_python_env\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\sakst\.conda\envs\vestas_python_env\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


In [3]:

project_id = os.environ.get("GOOGLE_CLOUD_PROJECT") or "intelligent-cc-optimization"
location = "us"
processor_id = os.environ.get("DOCUMENT_AI_PROCESSOR_ID") or "3ccf5d1678ae6859"
bucket_name = "my_storage_files"
file_name = "CC_project/CC_documents/AE_platinum_travel.pdf"
db_host = os.environ.get("DB_HOST") or "34.10.133.69" 
db_name = os.environ.get("DB_NAME") or "postgres"
db_user = os.environ.get("DB_USER") or "postgres"
db_pass = os.environ.get("DB_PASS") or "rag123"

In [ ]:
# def extract_document_ai_layout(project_id, location, processor_id, bucket_name, file_name):
#     """Connects to Document AI Layout Parser and returns extracted structural chunks."""
#     client_options = ClientOptions(api_endpoint=f"{location}-documentai.googleapis.com")
#     client = documentai.DocumentProcessorServiceClient(client_options=client_options)
    
#     processor_path = client.processor_path(project_id, location, processor_id)
#     # gcs_uri = f"gs://{bucket_name}/{file_name.lstrip('/')}".replace("//", "/")
#     gcs_uri=f"gs://{bucket_name}/{file_name}"
    
#     gcs_document = documentai.GcsDocument(gcs_uri=gcs_uri, mime_type="application/pdf")
#     request = documentai.ProcessRequest(name=processor_path, gcs_document=gcs_document)
    
#     print(f"Processing structural layout via Document AI for: {file_name}...")
#     result = client.process_document(request=request)
    
#     chunks = []
#     # Parse the modern document_layout blocks structure
#     if hasattr(result.document, 'document_layout') and result.document.document_layout.blocks:
#         for block in result.document.document_layout.blocks:
#             if hasattr(block, 'text_block') and block.text_block:
#                 text_content = block.text_block.text
#                 if text_content and text_content.strip():
#                     page_num = int(block.page_span.page_start) if block.page_span else 1
#                     chunks.append({
#                         "text": text_content.strip(),
#                         "page_number": page_num
#                     })
#     return chunks


def read_pdf_text_only(pdf_path: str) -> str:
    """
    Reads a PDF file and returns only its text content, 
    completely ignoring embedded images.
    """
    try:
        reader = PdfReader(pdf_path)
        full_text = []
        
        for i, page in enumerate(reader.pages):
            # extract_text() naturally skips images and extracts the text layer
            text = page.extract_text()
            if text:
                full_text.append(text)
                
        return "\n\n".join(full_text)
        
    except Exception as e:
        return f"Error reading PDF: {e}"

# Example Usage:
document_text = read_pdf_text_only(".\\data\\raw_pdfs\\HDFC_diners_black.pdf")
print(document_text)


CHUNK_SIZE_WORDS = 500
CHUNK_OVERLAP_WORDS = 75

def split_words_with_overlap(text: str, size: int, overlap: int) -> List[str]:
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + size
        chunk_words = words[start:end]
        chunks.append(" ".join(chunk_words))
        start = end - overlap

    return chunks

chunk_text = split_words_with_overlap(document_text, CHUNK_SIZE_WORDS, CHUNK_OVERLAP_WORDS)



def generate_vertex_embeddings(project_id, location, text_list):
    """Generates 768-dimensional dense vector embeddings using text-embedding-004."""
    # If the location passed is 'us', override it to 'us-central1' for Vertex AI compatibility
    vertex_location = "us-central1" if location == "us" else location
    vertex_location = "europe-west1" if location == "eu" else vertex_location
    
    # Initialize using the valid regional endpoint string
    vertexai.init(project=project_id, location=vertex_location)
    
    model = TextEmbeddingModel.from_pretrained("text-embedding-004")
    
    print(f"Generating vectors for {len(text_list)} text blocks via Vertex AI ({vertex_location})...")
    embeddings = model.get_embeddings(text_list)
    
    return [emb.values for emb in embeddings]

def store_in_postgres(db_config, metadata, chunks, vectors):
    """Safely stores chunks, custom card tracking attributes, and vector arrays."""
    conn = psycopg2.connect(
        host=db_config['host'],
        database=db_config['dbname'],
        user=db_config['user'],
        password=db_config['password'],
        port=db_config['port']
    )
    cursor = conn.cursor()
    
    insert_query = """
    INSERT INTO cc_reward_chunks (
        document_id, card_name, issuer, document_type, 
        page_number, effective_date, source_url, chunk_text, embedding
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s);
    """
    
    print(f"Inserting entries cleanly into PostgreSQL table...")
    try:
        for idx, chunk in enumerate(chunks):
            cursor.execute(insert_query, (
                metadata['document_id'],
                metadata['card_name'],
                metadata['issuer'],
                metadata['document_type'],
                chunk['page_number'],
                metadata['effective_date'],
                metadata['source_url'],
                chunk['text'],
                vectors[idx]  # psycopg2 handles list to array translations natively for pgvector
            ))
        conn.commit()
        print("🚀 Successfully saved database assets!")
    except Exception as e:
        conn.rollback()
        print(f"Error executing database transactions: {e}")
    finally:
        cursor.close()
        conn.close()

# --- CONFIGURATION EXECUTION BLOCK ---
if __name__ == "__main__":
    # 1. Core Environment Variables
    GCP_PROJECT = "intelligent-cc-optimization"
    GCP_LOCATION = "us" # Multi-region or regional endpoint matching your processor
    PROCESSOR_ID = "your_processor_hex_string_here"
    
    BUCKET_NAME = "my_storage_files"
    FILE_NAME = "CC_project/CC_documents/AE_platinum_travel.pdf"

    # 2. Extract Document Metadata Context attributes cleanly from naming conventions
    # e.g., 'AE_platinum_travel.pdf' -> issuer: 'AE', card_name: 'platinum_travel'
    base_file = os.path.basename(FILE_NAME)
    name_split = base_file.replace(".pdf", "").split("_", 1)
    
    inferred_issuer = name_split[0].upper() if len(name_split) > 0 else "UNKNOWN"
    inferred_card = name_split[1].replace("_", " ").title() if len(name_split) > 1 else "Generic Card"

    doc_metadata = {
        "document_id": f"DOC_{int(datetime.utcnow().timestamp())}", # Unique anchor ID string
        "card_name": inferred_card,
        "issuer": inferred_issuer,
        "document_type": "Terms and Conditions",
        "effective_date": "2026-01-01",  # Set static or parse via regex patterns
        "source_url": f"https://storage.googleapis.com/{bucket_name}/{file_name}".replace("//", "/")
    }

    # 3. Cloud SQL Connection Configurations
    db_credentials = {
        "host": db_host, # Your public IP or 127.0.0.1 if running through Cloud SQL Proxy
        "dbname": db_name,
        "user": db_user,
        "password": db_pass,
        "port": "5432"
    }

    # --- PIPELINE WORKFLOW EXECUTION ---
    # Step A: Parse PDF layout chunks
    extracted_chunks = extract_document_ai_layout(project_id, location, processor_id, bucket_name, file_name)
    
    if extracted_chunks:
        # Step B: Generate embeddings
        texts_to_embed = [item['text'] for item in extracted_chunks]
        computed_vectors = generate_vertex_embeddings(project_id, location, texts_to_embed)
        
        # Step C: Save to PostgreSQL Database
        store_in_postgres(db_credentials, doc_metadata, extracted_chunks, computed_vectors)
    else:
        print("Pipeline aborted: Document text parsing step returned empty layout sequences.")

Processing structural layout via Document AI for: CC_project/CC_documents/AE_platinum_travel.pdf...
Generating vectors for 9 text blocks via Vertex AI (us-central1)...
Inserting entries cleanly into PostgreSQL table...
🚀 Successfully saved database assets!


In [5]:
doc_metadata

{'document_id': 'DOC_1783814192',
 'card_name': 'Platinum Travel',
 'issuer': 'AE',
 'document_type': 'Terms and Conditions',
 'effective_date': '2026-01-01',
 'source_url': 'https:/storage.googleapis.com/my_storage_files/CC_project/CC_documents/AE_platinum_travel.pdf'}

In [8]:
extracted_chunks

[{'text': 'AMERICAN EXPRESS', 'page_number': 1},
 {'text': 'Terms and Conditions:', 'page_number': 1},
 {'text': 'GENERAL TERMS AND CONDITIONS', 'page_number': 1},
 {'text': 'PRODUCT BENEFIT TERMS AND CONDITIONS', 'page_number': 1},
 {'text': 'PRIORITY PASS TERMS AND CONDITIONS', 'page_number': 2},
 {'text': 'PARTNER TERMS AND CONDITIONS Taj, SeleQtions, and Vivanta hotels',
  'page_number': 2},
 {'text': 'Redemption Process', 'page_number': 2},
 {'text': 'Terms and Conditions', 'page_number': 2},
 {'text': 'Airport Lounges', 'page_number': 3}]

In [ ]:
process_document_sample

In [8]:
process_document_sample(
  project_id=project_id,
  location=location,
  processor_id=processor_id,
  file_path= f"gs://{bucket_name}/{file_name}",
  mime_type="application/pdf",
)

OSError: [Errno 22] Invalid argument: 'gs://my_storage_files/CC_project/CC_documents/AE_platinum_travel.pdf'

In [ ]:
import os
file_path_local = ".\data\raw_pdfs\"  # Local path for testing
file_name = "HDFC_diners_black.pdf"
local_files = [
    os.path.join(file_path_local, file_name)
    for file_name in os.listdir(file_path_local)
    if os.path.isfile(os.path.join(file_path_local, file_name))
]

SyntaxError: unterminated string literal (detected at line 2) (945766827.py, line 2)